# 6.24 - FACE hyperparameter heuristic diagnostics

Start small: this notebook only loads the libraries and the tabular datasets used by `benchmark_full.yaml`.


In [7]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent

if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from counterfactuals.datasets.loaders import (
    AdultDataset,
    CompasDataset,
    GermanCreditDataset,
    GiveMeSomeCreditDataset,
    HELOCDataset,
    LendingClubDataset,
    WisconsinBreastCancerDataset,
)
from counterfactuals.utils.config import read_yaml

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)


In [8]:
BENCHMARK_CFG = ROOT / 'configs' / 'benchmarks' / 'benchmark_full.yaml'
CFG = read_yaml(BENCHMARK_CFG)

DATASET_LOADERS = {
    'adult': AdultDataset,
    'compas': CompasDataset,
    'german_credit': GermanCreditDataset,
    'heloc': HELOCDataset,
    'give_me_some_credit': GiveMeSomeCreditDataset,
    'lending_club': LendingClubDataset,
    'wisconsin_breast_cancer': WisconsinBreastCancerDataset,
}

DATASET_NAMES = [d['name'] for d in CFG['datasets']]
SUBSAMPLE_N = 10000
SUBSAMPLE_SEED = 42
FACE_NORM = 1
TARGET_LARGEST_COMPONENT_RATIO = 0.90
MAX_ISOLATED_FRACTION = 0.05
MIN_AVG_DEGREE = 5.0
MAX_AVG_DEGREE = 100.0
DATASET_NAMES


['wisconsin_breast_cancer',
 'adult',
 'compas',
 'german_credit',
 'heloc',
 'give_me_some_credit',
 'lending_club']

In [9]:
rng = np.random.default_rng(SUBSAMPLE_SEED)
bundles = {}
rows = []

for dataset_name in DATASET_NAMES:
    loader = DATASET_LOADERS[dataset_name](data_dir=str(ROOT / 'data'), seed=42)
    loader.load()
    x_train, y_train = loader.get_train()
    x_test, y_test = loader.get_test()
    n_take = min(SUBSAMPLE_N, len(x_train))
    sample_idx = np.sort(rng.choice(len(x_train), size=n_take, replace=False))
    x_train_sample = x_train[sample_idx]
    y_train_sample = y_train[sample_idx]
    bundles[dataset_name] = {
        'x_train_full': x_train,
        'y_train_full': y_train,
        'x_train': x_train_sample,
        'y_train': y_train_sample,
        'x_test': x_test,
        'y_test': y_test,
        'sample_idx': sample_idx,
    }
    rows.append({
        'dataset': dataset_name,
        'n_train_full': int(len(x_train)),
        'n_train_sample': int(len(x_train_sample)),
        'n_test': int(len(x_test)),
        'n_features': int(x_train.shape[1]),
        'class0_train_fraction': float(np.mean(y_train_sample == 0)),
        'class1_train_fraction': float(np.mean(y_train_sample == 1)),
    })

DATASET_SUMMARY_DF = pd.DataFrame(rows).sort_values('dataset').reset_index(drop=True)
display(DATASET_SUMMARY_DF)


,dataset,n_train_full,n_train_sample,n_test,n_features,class0_train_fraction,class1_train_fraction
0,adult,36178,10000,4522,104,0.745600,0.254400
1,compas,4938,4938,617,14,0.547388,0.452612
2,german_credit,800,800,100,61,0.692500,0.307500
3,give_me_some_credit,120000,10000,15000,10,0.934400,0.065600
4,heloc,8369,8369,1045,23,0.476640,0.523360
5,lending_club,30823,10000,3852,25,0.853900,0.146100
6,wisconsin_breast_cancer,457,457,56,30,0.619256,0.380744


In [10]:
from scipy.sparse.csgraph import connected_components
from sklearn.neighbors import radius_neighbors_graph

# EPSILON_GRID = list(np.arange(0.1, 5.1, 0.1)) # Worked for L2
EPSILON_GRID = np.arange(3.5, 25.5, 0.5)

def sklearn_metric_kwargs(norm: int | float | str) -> dict:
    if norm == 1:
        return {'metric': 'minkowski', 'p': 1}
    if norm == 2:
        return {'metric': 'minkowski', 'p': 2}
    if norm in {'inf', np.inf}:
        return {'metric': 'chebyshev'}
    raise ValueError('Supported norms are 1, 2, and inf.')

def graph_metrics(x: np.ndarray, epsilon: float, norm: int | float | str) -> dict:
    adj = radius_neighbors_graph(
        x,
        radius=float(epsilon),
        mode="connectivity",
        include_self=False,
        **sklearn_metric_kwargs(norm),
    )
    degrees = np.asarray(adj.getnnz(axis=1)).astype(np.int64)
    n_components, labels = connected_components(adj, directed=False, return_labels=True)
    comp_sizes = np.bincount(labels, minlength=n_components) if n_components > 0 else np.array([], dtype=np.int64)
    largest_component = int(comp_sizes.max()) if comp_sizes.size else 0
    isolated_nodes = int(np.sum(degrees == 0)) if len(degrees) else 0
    return {
        "norm": str(norm),
        "epsilon": float(epsilon),
        "n_nodes": int(x.shape[0]),
        "n_edges": int(adj.nnz // 2),
        "n_components": int(n_components),
        "isolated_nodes": isolated_nodes,
        "isolated_fraction": float(isolated_nodes / x.shape[0]) if x.shape[0] else 0.0,
        "largest_component": largest_component,
        "largest_component_ratio": float(largest_component / x.shape[0]) if x.shape[0] else 0.0,
        "avg_degree": float(degrees.mean()) if len(degrees) else 0.0,
        "median_degree": float(np.median(degrees)) if len(degrees) else 0.0,
        "max_degree": int(degrees.max()) if len(degrees) else 0,
    }


In [11]:
graph_rows = []

for dataset_name, bundle in bundles.items():
    x = bundle["x_train"]
    for epsilon in EPSILON_GRID:
        row = {"dataset": dataset_name}
        row.update(graph_metrics(x, epsilon, FACE_NORM))
        graph_rows.append(row)

eps_ok = lambda row: (
    (row['largest_component_ratio'] >= TARGET_LARGEST_COMPONENT_RATIO)
    and (row['isolated_fraction'] <= MAX_ISOLATED_FRACTION)
    and (row['avg_degree'] >= MIN_AVG_DEGREE)
    and (row['avg_degree'] <= MAX_AVG_DEGREE)
)
GRAPH_METRICS_DF = pd.DataFrame(graph_rows).sort_values(["dataset", "epsilon"]).reset_index(drop=True)
GRAPH_METRICS_DF['eps_ok'] = GRAPH_METRICS_DF.apply(eps_ok, axis=1)
GRAPH_METRICS_DF = GRAPH_METRICS_DF[[
    'dataset', 'norm', 'eps_ok', 'epsilon', 'n_nodes', 'n_edges', 'n_components',
    'isolated_nodes', 'isolated_fraction', 'largest_component',
    'largest_component_ratio', 'avg_degree', 'median_degree', 'max_degree'
]]
display(GRAPH_METRICS_DF)


,dataset,norm,eps_ok,epsilon,n_nodes,n_edges,n_components,isolated_nodes,isolated_fraction,largest_component,largest_component_ratio,avg_degree,median_degree,max_degree
0,adult,1,False,3.5,10000,226188,1815,1685,0.168500,7811,0.781100,45.237600,12.0,528
1,adult,1,False,4.0,10000,352656,1347,1246,0.124600,8472,0.847200,70.531200,21.0,701
2,adult,1,False,4.5,10000,513786,915,864,0.086400,8974,0.897400,102.757200,35.0,881
3,adult,1,False,5.0,10000,735141,605,580,0.058000,9331,0.933100,147.028200,59.5,1064
4,adult,1,False,5.5,10000,1036694,408,391,0.039100,9539,0.953900,207.338800,98.0,1307
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
303,wisconsin_breast_cancer,1,False,23.0,457,37023,13,12,0.026258,445,0.973742,162.026258,189.0,298
304,wisconsin_breast_cancer,1,False,23.5,457,38659,13,12,0.026258,445,0.973742,169.185996,197.0,308
305,wisconsin_breast_cancer,1,False,24.0,457,40215,9,8,0.017505,449,0.982495,175.995624,209.0,312
306,wisconsin_breast_cancer,1,False,24.5,457,41812,8,7,0.015317,450,0.984683,182.984683,217.0,319


In [12]:
MIN_EPSILON_DF = (
    GRAPH_METRICS_DF[GRAPH_METRICS_DF['eps_ok']]
    .sort_values(['dataset', 'epsilon'])
    .groupby('dataset', as_index=False)
    .first()[['dataset', 'norm', 'epsilon', 'avg_degree', 'isolated_fraction', 'largest_component_ratio', 'n_components']]
    .rename(columns={'epsilon': 'min_valid_epsilon'})
)
display(MIN_EPSILON_DF.round(2))


,dataset,norm,min_valid_epsilon,avg_degree,isolated_fraction,largest_component_ratio,n_components
0,german_credit,1,12.0,25.93,0.04,0.96,35
1,lending_club,1,4.0,99.39,0.04,0.91,412
2,wisconsin_breast_cancer,1,18.5,95.89,0.05,0.95,23
